In [ ]:
import os
os.environ["GROQ_API_KEY"]="your api key"

In [ ]:
!pip install langchain-core langchain-groq requests

In [ ]:
import requests
from langchain_core.tools import tool

In [ ]:
# tools creation
from langchain_core.tools import InjectedToolArg
from typing import Annotated
# Tool-1

@tool
def get_conversion_factor(basecurrency: str, targetcurrency: str)->float:
  """
  This tools fetches the currency conversion factor between a given base currency ans a target currency
  """
  # we will be using exchangerateapi for gettingthese coversions in realtime
  url =f"https://v6.exchangerate-api.com/v6/00516ddea6b33c61908ee175/pair/{basecurrency}/{targetcurrency}"

  response = requests.get(url)

  return response.json()["conversion_rate"]

# Tool-2

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg])->float:
  """
  Given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate


In [ ]:
# lets test it
get_conversion_factor.invoke({"basecurrency": "USD", "targetcurrency":"INR"})

94.4113

In [ ]:
convert.invoke({"base_currency_value":10,"conversion_rate": 94.4113})

944.1129999999999

In [ ]:
# tool binding
from langchain_groq import ChatGroq
LLM_Model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)

In [ ]:
LLM_Binding = LLM_Model.bind_tools([get_conversion_factor,convert])

In [ ]:
#tool calling
from langchain_core.messages import HumanMessage
messages = [HumanMessage("what is the conversion factor between USD and INR and after that based on that can you convert 100 usd to INR ")]

In [ ]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and after that based on that can you convert 100 usd to INR ', additional_kwargs={}, response_metadata={})]

In [ ]:
ai_Message = LLM_Binding.invoke(messages)

In [ ]:
messages.append(ai_Message)

In [ ]:
ai_Message = LLM_Binding.invoke(messages)

In [ ]:
ai_Message

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get conversion factor, then compute conversion. Use get_conversion_factor, then convert using that factor? Actually convert function expects base_currency_value, but also likely uses the factor? The description: "Given a currency conversion rate this function calculates the target currency value from a given base currency value". But we need to provide conversion rate? The function signature only takes base_currency_value, no rate. Possibly the function uses the last fetched conversion factor internally? Might need to call convert with base_currency_value:100. Let\'s first call get_conversion_factor.', 'tool_calls': [{'id': 'fc_e1d45908-3722-47d8-8e90-f730316bfc8f', 'function': {'arguments': '{"basecurrency":"USD","targetcurrency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 223, 'total_tokens': 365, 'completion_tim

In [ ]:
for tool_call in ai_Message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message_1 = get_conversion_factor.invoke(tool_call)
    print(tool_message_1)
    conversion_rate = tool_message_1.content
    # append this tool message to message list
    messages.append(tool_message_1)
  # # execute the 2nd tool using the conversion rate from tool 1
  # if tool_call['name'] == 'convert':
  #   tool_message_2 = convert.invoke({"base_currency_value":100,"conversion_rate": conversion_rate})
  #   print(tool_message_2)

content='94.4113' name='get_conversion_factor' tool_call_id='fc_e1d45908-3722-47d8-8e90-f730316bfc8f'
